# Lesson 18: 使用加密收据保护 AI 代理

## 实践笔记本

本笔记本包含四个练习：

1. <strong>签署您的第一个收据</strong>，用于代理工具调用并验证它。
2. <strong>篡改收据</strong>，观察验证失败。
3. <strong>构建三重收据链</strong>，确认链的完整性。
4. **包装 Microsoft 代理框架工具调用**，使每个操作都发出收据。

所有加密原语均来自维护良好的库（使用 `pynacl` 实现 Ed25519，`jcs` 实现 RFC 8785 规范 JSON，Python 标准库中的 `hashlib` 实现 SHA-256）。收据逻辑本身是普通的 Python 代码，您可以阅读和修改。

请按顺序运行代码单元。每个部分简短且独立。


## 设置

安装这两个依赖项。两者都具有宽松的许可证（Apache-2.0 / MIT）。


In [ ]:
!pip install -q pynacl jcs

In [ ]:
import json
import hashlib
import base64
from datetime import datetime, timezone

from nacl import signing
from nacl.exceptions import BadSignatureError
from jcs import canonicalize

## 辅助工具

这两个辅助工具处理 base64url 编码（无填充）和任意对象的 SHA-256 哈希。它们使笔记本的其余部分专注于收据逻辑本身。


In [ ]:
def b64url_nopad(data: bytes) -> str:
    """Base64url-encode bytes without padding (RFC 4648 Section 5)."""
    return base64.urlsafe_b64encode(data).decode("ascii").rstrip("=")

def b64url_decode(s: str) -> bytes:
    """Decode a base64url string that may be missing padding."""
    padding = "=" * ((4 - len(s) % 4) % 4)
    return base64.urlsafe_b64decode(s + padding)

def sha256_canonical(obj) -> str:
    """
    SHA-256 hash of a Python object, computed over its JCS-canonical JSON form.
    Returns a 'sha256:' prefixed hex digest so callers can identify the algorithm.
    """
    canonical = canonicalize(obj)
    digest = hashlib.sha256(canonical).hexdigest()
    return f"sha256:{digest}"

## Section 1: 签署您的第一张收据

想象一下，我们的<strong>Contoso Travel</strong>代理刚刚为客户查询了从悉尼到洛杉矶的航班。我们希望将此工具调用记录为已签名的收据，以便未来的审计人员可以在不信任我们的情况下进行验证。

### 第1.1步：生成签名密钥

在生产环境中，代理的签名密钥将存放在硬件安全模块（HSM）、Azure Key Vault或类似的受保护存储中。对于本课程，我们在内存中生成一个新的密钥。


In [ ]:
signing_key = signing.SigningKey.generate()
verify_key = signing_key.verify_key

public_key_b64 = b64url_nopad(bytes(verify_key))
print(f"Public key (Ed25519, 32 bytes): {public_key_b64}")

### 第1.2步：构建收据负载

负载包含了我们希望收据证明的所有内容：谁执行了操作，使用了什么工具，带了哪些参数，返回了什么，遵循了什么策略，以及何时执行。我们对参数和结果进行哈希处理，而不是直接包含它们，以防止收据泄露敏感内容。


In [ ]:
tool_args = {
    "origin": "SYD",
    "destination": "LAX",
    "departure_date": "2026-06-15",
    "passengers": 2,
}

tool_result = [
    {"flight": "QF11", "price": 1850, "stops": 0},
    {"flight": "UA864", "price": 1620, "stops": 1},
    {"flight": "DL11", "price": 1740, "stops": 0},
]

payload = {
    "type": "agent.tool_call.v1",
    "agent_id": "contoso-travel-bot",
    "tool_name": "lookup_flights",
    "tool_args_hash": sha256_canonical(tool_args),
    "result_hash": sha256_canonical(tool_result),
    "policy_id": "contoso-travel-policy-v3",
    "timestamp": datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
    "sequence": 0,
    "previous_receipt_hash": None,
}

print(json.dumps(payload, indent=2))

### 第1.3步：签署并组装回执

三个步骤：

1. 使用JCS对有效负载进行规范化，以便两个实现产生相同的逻辑回执，从而生成字节完全相同的字节。
2. 使用SHA-256对规范化的字节进行哈希。
3. 使用Ed25519私钥对哈希进行签名。

然后将签名附加到原始有效负载上，生成最终回执。


In [ ]:
def sign_receipt(payload: dict, signing_key: signing.SigningKey, verify_key) -> dict:
    """
    Sign a receipt payload. Returns the receipt with attached signature and public key.
    The 'signature' and 'public_key' fields are NOT part of the canonical signed bytes.
    """
    canonical = canonicalize(payload)
    message_hash = hashlib.sha256(canonical).digest()
    signature_bytes = signing_key.sign(message_hash).signature
    return {
        **payload,
        "signature": {
            "alg": "EdDSA",
            "sig": b64url_nopad(signature_bytes),
            "public_key": b64url_nopad(bytes(verify_key)),
        },
    }

receipt = sign_receipt(payload, signing_key, verify_key)
print(json.dumps(receipt, indent=2))

### 第1.4步：验证收据

验证过程是逆向操作。我们去除签名，重新计算规范哈希，然后根据收据中的公钥检查签名。

执行此验证的审计员不需要我们提供任何东西，除了收据本身。无需调用服务，无需查询密钥目录，无需信任。


In [ ]:
def verify_receipt(receipt: dict) -> bool:
    """
    Verify a receipt's Ed25519 signature.
    Returns True if valid, False otherwise.
    """
    sig_obj = receipt.get("signature")
    if not sig_obj or sig_obj.get("alg") != "EdDSA":
        return False

    # Reconstruct the payload that was actually signed (everything except signature).
    payload = {k: v for k, v in receipt.items() if k != "signature"}

    canonical = canonicalize(payload)
    message_hash = hashlib.sha256(canonical).digest()

    try:
        verify_key = signing.VerifyKey(b64url_decode(sig_obj["public_key"]))
        verify_key.verify(message_hash, b64url_decode(sig_obj["sig"]))
        return True
    except BadSignatureError:
        return False
    except Exception as exc:
        print(f"Verification error: {exc}")
        return False

is_valid = verify_receipt(receipt)
print(f"Receipt is valid: {is_valid}")

你应该看到 `Receipt is valid: True`。代理已经生成了它的第一个经过密码学签名的审计记录。


## 第二部分：篡改收据

收据的全部意义在于它们具有防篡改性。让我们验证一下。

我们将修改收据中的一个字符，然后观察验证失败。


In [ ]:
import copy

tampered = copy.deepcopy(receipt)

# Modify the policy_id field (this is what an attacker might do to claim
# the action was governed by a more permissive policy than was actually used).
original_policy = tampered["policy_id"]
tampered["policy_id"] = "contoso-travel-policy-PERMISSIVE"

print(f"Original policy_id:  {original_policy}")
print(f"Tampered policy_id:  {tampered['policy_id']}")
print()
print(f"Tampered receipt valid? {verify_receipt(tampered)}")

### 刚刚发生了什么？

当我们更改 `policy_id` 时，规范字节发生了变化。那些字节的 SHA-256 哈希也发生了变化。签名（它是基于原始哈希的）不再与新的哈希匹配。验证正确地返回了 `False`。

除非攻击者拥有私钥，否则无法修改收据的任何字段后还能通过验证。只要私钥保存在密钥库中且公钥已发布，篡改就不可能被隐藏。

自己试试：修改上面单元格中的 `tool_name`、`agent_id` 或 `timestamp` 并重新运行。每次更改都会生成无效的收据。


## 第3节：将收据链接在一起

单个收据保护一个操作。大多数代理执行多个操作。为了使整个序列具有防篡改性，我们通过在新收据的负载中包含前一个收据的哈希来将每个收据链接到前一个收据。

```text
Receipt 0  -->  Receipt 1  -->  Receipt 2
                  |                 |
                  +-- previous_receipt_hash field --+
```

如果有人删除或重新排序收据，链条将在该点断开。验证任何后续收据都会失败，因为其 `previous_receipt_hash` 不再与前一个收据的实际哈希匹配。


In [ ]:
def receipt_hash(receipt: dict) -> str:
    """
    Compute the chain hash of a complete receipt (including signature).
    This becomes the previous_receipt_hash of the next receipt in the chain.
    """
    canonical = canonicalize(receipt)
    digest = hashlib.sha256(canonical).hexdigest()
    return f"sha256:{digest}"

def make_receipt(
    tool_name: str,
    tool_args: dict,
    tool_result,
    sequence: int,
    previous_receipt_hash,
    signing_key,
    verify_key,
    agent_id: str = "contoso-travel-bot",
    policy_id: str = "contoso-travel-policy-v3",
) -> dict:
    """Convenience: build, sign, and return a receipt for one tool call."""
    payload = {
        "type": "agent.tool_call.v1",
        "agent_id": agent_id,
        "tool_name": tool_name,
        "tool_args_hash": sha256_canonical(tool_args),
        "result_hash": sha256_canonical(tool_result),
        "policy_id": policy_id,
        "timestamp": datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
        "sequence": sequence,
        "previous_receipt_hash": previous_receipt_hash,
    }
    return sign_receipt(payload, signing_key, verify_key)

In [ ]:
# Build a chain of three receipts: search, hold, book.
r0 = make_receipt(
    tool_name="lookup_flights",
    tool_args={"origin": "SYD", "destination": "LAX", "date": "2026-06-15"},
    tool_result=[{"flight": "QF11", "price": 1850}],
    sequence=0,
    previous_receipt_hash=None,
    signing_key=signing_key,
    verify_key=verify_key,
)

r1 = make_receipt(
    tool_name="hold_seat",
    tool_args={"flight": "QF11", "seat": "14A", "hold_minutes": 30},
    tool_result={"hold_id": "H8472", "expires_at": "2026-06-15T15:00:00Z"},
    sequence=1,
    previous_receipt_hash=receipt_hash(r0),
    signing_key=signing_key,
    verify_key=verify_key,
)

r2 = make_receipt(
    tool_name="confirm_booking",
    tool_args={"hold_id": "H8472", "payment_token": "tok_redacted"},
    tool_result={"booking_ref": "CT-09182", "status": "confirmed"},
    sequence=2,
    previous_receipt_hash=receipt_hash(r1),
    signing_key=signing_key,
    verify_key=verify_key,
)

chain = [r0, r1, r2]
for i, r in enumerate(chain):
    print(f"Receipt {i}: tool={r['tool_name']}, prev={r['previous_receipt_hash']}")

In [ ]:
def verify_chain(chain: list) -> list[dict]:
    """
    Verify a sequence of receipts:
      1. Each receipt's signature must verify.
      2. Each receipt (except the genesis) must reference the previous receipt's hash.
      3. Sequence numbers must match each receipt's zero-based position in the chain.
    Returns a list of per-receipt result dicts.
    """
    results = []
    for i, receipt in enumerate(chain):
        sig_ok = verify_receipt(receipt)

        if i == 0:
            chain_ok = receipt["previous_receipt_hash"] is None
        else:
            expected = receipt_hash(chain[i - 1])
            chain_ok = receipt["previous_receipt_hash"] == expected

        seq_ok = receipt["sequence"] == i

        results.append({
            "index": i,
            "tool": receipt["tool_name"],
            "signature_valid": sig_ok,
            "chain_link_valid": chain_ok,
            "sequence_valid": seq_ok,
            "overall_valid": sig_ok and chain_ok and seq_ok,
        })
    return results

for r in verify_chain(chain):
    status = "VALID" if r["overall_valid"] else "INVALID"
    print(f"Receipt {r['index']} ({r['tool']:>18}): {status}")

现在通过篡改中间的收据来破坏链条并重新验证。被篡改的收据未通过其签名检查，且下一个收据未通过其链条链接检查（因为其 `previous_receipt_hash` 不再匹配被修改的中间收据的哈希）。


In [ ]:
# Tamper with the middle receipt: change the hold duration to something
# more permissive than was actually authorized.
tampered_chain = [copy.deepcopy(r) for r in chain]
tampered_chain[1]["tool_args_hash"] = sha256_canonical(
    {"flight": "QF11", "seat": "14A", "hold_minutes": 9999}
)

for r in verify_chain(tampered_chain):
    status = "VALID" if r["overall_valid"] else "INVALID"
    why = ""
    if not r["overall_valid"]:
        reasons = []
        if not r["signature_valid"]:
            reasons.append("signature")
        if not r["chain_link_valid"]:
            reasons.append("chain link")
        if not r["sequence_valid"]:
            reasons.append("sequence")
        why = " (failed: " + ", ".join(reasons) + ")"
    print(f"Receipt {r['index']} ({r['tool']:>18}): {status}{why}")

收据 0 仍然验证通过（它未被修改且没有依赖的前序收据）。收据 1 由于我们更改了 `tool_args_hash`，导致其签名校验失败。收据 2 由于其 `previous_receipt_hash` 是基于原始（现已被修改的）收据 1 计算的，因此链条链接校验失败。

即使攻击者重新签署了被修改的收据 1（没有私钥他们无法做到这一点），收据 2 中的链条链接不匹配仍会暴露篡改行为。为了掩盖更改，攻击者必须从修改点开始重新签署所有收据，这需要拥有私钥。


## 第4节：用收据签名包装代理工具调用

在实际部署中，您不希望每个代理作者都记得调用 `make_receipt`。您希望每次工具调用都自动进行收据签名。

这里是最简单的模式：一个包装类，接受任何可调用的工具函数并返回一个可发出收据的版本。它可以适配任何代理框架，包括微软代理框架（`agent_framework.azure`）。

如果您尚未设置 Azure AI Foundry 项目，下面的本地模拟仍然演示了这一模式。


In [ ]:
class ReceiptedTool:
    """
    Wraps a tool function so every invocation produces a signed receipt.
    Receipts are appended to a chain held by this object.

    Accepts both positional and keyword arguments. The receipt's
    tool_args field records args (as a list) and kwargs (as a dict)
    so the canonical hash binds to whichever the caller supplied.
    """

    def __init__(self, name: str, fn, signing_key, verify_key, agent_id: str, policy_id: str):
        self.name = name
        self.fn = fn
        self.signing_key = signing_key
        self.verify_key = verify_key
        self.agent_id = agent_id
        self.policy_id = policy_id
        self.receipts: list = []

    def __call__(self, *args, **kwargs):
        result = self.fn(*args, **kwargs)
        previous_hash = receipt_hash(self.receipts[-1]) if self.receipts else None
        receipt = make_receipt(
            tool_name=self.name,
            tool_args={"args": list(args), "kwargs": kwargs},
            tool_result=result,
            sequence=len(self.receipts),
            previous_receipt_hash=previous_hash,
            signing_key=self.signing_key,
            verify_key=self.verify_key,
            agent_id=self.agent_id,
            policy_id=self.policy_id,
        )
        self.receipts.append(receipt)
        return result

In [ ]:
# Example tool: a mock flight lookup. In a real Microsoft Agent Framework deployment,
# this would be a function passed to AzureAIProjectAgentProvider as a tool.
def mock_lookup_flights(origin: str, destination: str, departure_date: str) -> list:
    return [
        {"flight": "QF11", "price": 1850, "stops": 0},
        {"flight": "UA864", "price": 1620, "stops": 1},
    ]

# Wrap it with receipt signing.
receipted_lookup = ReceiptedTool(
    name="lookup_flights",
    fn=mock_lookup_flights,
    signing_key=signing_key,
    verify_key=verify_key,
    agent_id="contoso-travel-bot",
    policy_id="contoso-travel-policy-v3",
)

# Use the wrapped tool exactly like the original.
results_a = receipted_lookup(origin="SYD", destination="LAX", departure_date="2026-06-15")
results_b = receipted_lookup(origin="SYD", destination="NRT", departure_date="2026-07-02")
results_c = receipted_lookup(origin="MEL", destination="SIN", departure_date="2026-08-10")

print(f"Tool was called {len(receipted_lookup.receipts)} times.")
print(f"Each call produced a signed receipt linked to the previous one.")
print()

for r in verify_chain(receipted_lookup.receipts):
    status = "VALID" if r["overall_valid"] else "INVALID"
    print(f"Receipt {r['index']} ({r['tool']}): {status}")

### 与 Microsoft Agent Framework 集成

上面的 `ReceiptedTool` 包装器是与框架无关的。要在使用 Microsoft Agent Framework 构建的代理中使用它，请将包装后的函数注册为工具。以下是一个示例（您需要用真实的 Azure AI Foundry 工具注册替换此模拟）：

```python
# 显示集成结构的伪代码。
# 来自 agent_framework.azure 的 AzureAIProjectAgentProvider
# 来自 azure.identity 的 AzureCliCredential
#
# provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())
# agent = provider.create_agent(
#     instructions="你是 Contoso 旅游代理 ..."
#     tools=[receipted_lookup],   # 包装过的工具，不是原始函数
# )
# response = agent.run("查找六月从悉尼到洛杉矶的航班。")
#
# # 运行后，代理调用的每个工具都有签名收据：
# audit_chain = receipted_lookup.receipts
```

代理框架无需了解任何有关收据的信息。收据签名是包装在工具周围的，而不是直接集成到框架中。这就是如何在不重写代理的情况下，为现有代理代码添加来源信息的方法。


## 回顾与拓展挑战

你已经：

- 生成了一个 Ed25519 密钥对。
- 构建并签署了一个代理工具调用的收据。
- 使用仅公钥的离线方式验证了收据。
- 篡改了收据并观察到验证失败。
- 构建了一个包含三个收据的哈希链序列。
- 篡改了链条的中间部分，观察到签名失败和链环失败。
- 用自动签署收据的方式包装了一个工具函数。

**拓展挑战。** 扩展收据模式，添加一个 `request_id` 字段（用于分布式追踪的 UUID）。更新 `make_receipt` 以包含该字段，并确认收据仍能端到端验证。然后在签署后修改该字段并确认验证失败。这将迫使你深入理解规范编码的每个字节如何影响签名。

**重要界限。** 收据证明三件事，也只证明这三件事：归属（此密钥签署了此内容）、完整性（内容自签署后未被更改）、顺序（此收据发生在另一收据之后）。它们<strong>不</strong>证明代理的行为是正确的，不证明 `policy_id` 中指定的策略确实被评估，也不证明代理完全遵守了所有规则。收据是基础，治理体系是你建立在其上的系统。

请再次带着这一界限阅读课程 README。团队对收据最常见的误解是认为“我们有收据”就意味着“我们已受到治理”。事实并非如此。收据使得代理行为可审计，却不保证其正确性。


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**免责声明**：
本文件由 AI 翻译服务 [Co-op Translator](https://github.com/Azure/co-op-translator) 翻译完成。尽管我们力求准确，但请注意，自动翻译可能包含错误或不准确之处。原始语言版文件应视为权威来源。对于重要信息，建议使用专业人工翻译。我们对因使用本翻译而产生的任何误解或误释不承担责任。
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
